# FairFace DPO Training

This notebook snapshots the current FairFace trainer, runs one timestamped training job, stores all run artifacts under `outputs/`, and copies the final LoRA adapter into `lora_weights/` without overwriting prior runs.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import re
import shutil
import subprocess
import sys
import time

ROOT_CANDIDATES = [
    Path('/workspace/fairface_latest_code'),
    Path.cwd(),
    Path.cwd() / 'fairface_latest_code',
]
ROOT = next((p for p in ROOT_CANDIDATES if p.exists() and p.name == 'fairface_latest_code'), ROOT_CANDIDATES[0])
ROOT.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR = ROOT / 'outputs'
LORA_WEIGHTS_DIR = ROOT / 'lora_weights'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
LORA_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_SCRIPT_CANDIDATES = [
    ROOT.parent / 'analysis' / 'train_fairface_iat_dpo.py',
    Path('/workspace/analysis/train_fairface_iat_dpo.py'),
]
SOURCE_SCRIPT = next((p for p in SOURCE_SCRIPT_CANDIDATES if p.exists()), None)
if SOURCE_SCRIPT is None:
    raise FileNotFoundError('Could not find analysis/train_fairface_iat_dpo.py')
SNAPSHOT_SCRIPT = ROOT / 'train_fairface_iat_dpo_latest.py'
print('ROOT:', ROOT)
print('SOURCE_SCRIPT:', SOURCE_SCRIPT)


In [ ]:
REQUIRED = {
    'packaging': 'packaging>=24.2',
    'pandas': 'pandas==2.2.3',
    'PIL': 'pillow==11.1.0',
    'matplotlib': 'matplotlib==3.10.0',
    'transformers': 'transformers==4.50.0',
    'accelerate': 'accelerate==1.4.0',
    'peft': 'peft==0.14.0',
    'huggingface_hub': 'huggingface_hub==0.28.1',
    'datasets': 'datasets>=3.4.0',
    'sentencepiece': 'sentencepiece==0.2.0',
}

missing = []
for module_name, pip_spec in REQUIRED.items():
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(pip_spec)

if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
    importlib.invalidate_caches()

import pandas as pd
from IPython.display import Image, Markdown, display


In [ ]:
# Training configuration
RUN_LABEL = 'default'
TRAIN_SAMPLE_PER_GROUP = 2
EVAL_SAMPLE_PER_GROUP = 2
SEED = 7
TRAIN_SEED = None
EVAL_SEED = None
TEMPERATURE = 1.0
TOP_P = 1.0
MAX_NEW_TOKENS = 30
FALLBACK_MAX_NEW_TOKENS = 20
IAT_DPO_NUM_CANDIDATES = 8
IAT_DPO_PAIR_REPEAT = 4
IAT_DPO_REQUIRE_SCORE_MARGIN = 0.25
IAT_FORBIDDEN_PENALTY = 5.0
IAT_NOVELTY_WEIGHT = 2.0
IAT_VALENCE_CENTER_PENALTY = 0.4
DPO_BETA = 0.05
DPO_LR = 5e-5
DPO_WARMUP_STEPS = 8
DPO_GRAD_ACCUM = 4
DPO_MAX_STEPS = 16
EVAL_REPEATS = 10
PRECOMPUTE_DPO_REFERENCE = True
REUSE_TRAIN_METADATA_CSV = None
REUSE_EVAL_METADATA_CSV = None
EXTRA_ARGS = []

# Example for the larger April 15-style run:
# TRAIN_SAMPLE_PER_GROUP = 6
# EVAL_SAMPLE_PER_GROUP = 6
# SEED = 17
# TRAIN_SEED = 17
# EVAL_SEED = 1017
# IAT_DPO_PAIR_REPEAT = 6
# IAT_DPO_REQUIRE_SCORE_MARGIN = 0.0
# DPO_MAX_STEPS = 48


In [ ]:
def slugify(value: str) -> str:
    value = re.sub(r'[^A-Za-z0-9._-]+', '_', str(value).strip())
    return value.strip('._-') or 'run'

def make_unique_run_name(prefix: str, label: str) -> str:
    timestamp = time.strftime('%Y%m%d_%H%M%S')
    base = f"{prefix}_{slugify(label)}_{timestamp}" if label else f"{prefix}_{timestamp}"
    candidate = base
    counter = 2
    while (OUTPUTS_DIR / candidate).exists() or (LORA_WEIGHTS_DIR / candidate).exists():
        candidate = f"{base}_{counter:02d}"
        counter += 1
    return candidate

RUN_NAME = make_unique_run_name('fairface_dpo', RUN_LABEL)
RUN_DIR = OUTPUTS_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)
LOG_PATH = RUN_DIR / 'training_notebook.log'
shutil.copy2(SOURCE_SCRIPT, SNAPSHOT_SCRIPT)

cmd = [
    sys.executable,
    str(SNAPSHOT_SCRIPT),
    '--output-dir', str(RUN_DIR),
    '--train-sample-per-group', str(TRAIN_SAMPLE_PER_GROUP),
    '--eval-sample-per-group', str(EVAL_SAMPLE_PER_GROUP),
    '--seed', str(SEED),
    '--temperature', str(TEMPERATURE),
    '--top-p', str(TOP_P),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--fallback-max-new-tokens', str(FALLBACK_MAX_NEW_TOKENS),
    '--iat-dpo-num-candidates', str(IAT_DPO_NUM_CANDIDATES),
    '--iat-dpo-pair-repeat', str(IAT_DPO_PAIR_REPEAT),
    '--iat-dpo-require-score-margin', str(IAT_DPO_REQUIRE_SCORE_MARGIN),
    '--iat-forbidden-penalty', str(IAT_FORBIDDEN_PENALTY),
    '--iat-novelty-weight', str(IAT_NOVELTY_WEIGHT),
    '--iat-valence-center-penalty', str(IAT_VALENCE_CENTER_PENALTY),
    '--dpo-beta', str(DPO_BETA),
    '--dpo-lr', str(DPO_LR),
    '--dpo-warmup-steps', str(DPO_WARMUP_STEPS),
    '--dpo-grad-accum', str(DPO_GRAD_ACCUM),
    '--dpo-max-steps', str(DPO_MAX_STEPS),
    '--eval-repeats', str(EVAL_REPEATS),
]
if TRAIN_SEED is not None:
    cmd.extend(['--train-seed', str(TRAIN_SEED)])
if EVAL_SEED is not None:
    cmd.extend(['--eval-seed', str(EVAL_SEED)])
if REUSE_TRAIN_METADATA_CSV:
    cmd.extend(['--reuse-train-metadata-csv', str(REUSE_TRAIN_METADATA_CSV)])
if REUSE_EVAL_METADATA_CSV:
    cmd.extend(['--reuse-eval-metadata-csv', str(REUSE_EVAL_METADATA_CSV)])
cmd.append('--precompute-dpo-reference' if PRECOMPUTE_DPO_REFERENCE else '--no-precompute-dpo-reference')
cmd.extend([str(item) for item in EXTRA_ARGS])

env = os.environ.copy()
env.setdefault('PYTHONUNBUFFERED', '1')
env.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
Path(env['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

print('Running command:')
print(' '.join(cmd))
with open(LOG_PATH, 'w', encoding='utf-8') as log_file:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f'Training failed with exit code {return_code}. Check {LOG_PATH}')

manifest_path = RUN_DIR / 'fairface_train_manifest.json'
if not manifest_path.exists():
    raise FileNotFoundError(f'Manifest missing at {manifest_path}')
manifest = json.loads(manifest_path.read_text())

final_adapter_dir = RUN_DIR / 'checkpoints' / 'final'
if not final_adapter_dir.exists():
    raise FileNotFoundError(f'Final adapter missing at {final_adapter_dir}')
LORA_DIR = LORA_WEIGHTS_DIR / RUN_NAME
shutil.copytree(final_adapter_dir, LORA_DIR)

summary = {
    'run_name': RUN_NAME,
    'created_at': manifest.get('created_at'),
    'output_dir': str(RUN_DIR),
    'lora_dir': str(LORA_DIR),
    'seed': manifest.get('seed'),
    'train_sample_per_group': manifest.get('train_sample_per_group'),
    'eval_sample_per_group': manifest.get('eval_sample_per_group'),
    'training_pairs_after_repeat': manifest.get('training_pairs_after_repeat'),
    'dpo_max_steps': manifest.get('dpo_max_steps'),
    'base_neg_gap_abs': (manifest.get('base_eval_metrics') or {}).get('neg_gap_abs'),
    'adapter_neg_gap_abs': (manifest.get('adapter_eval_metrics') or {}).get('neg_gap_abs'),
    'base_positive_gap_abs': (manifest.get('base_eval_metrics') or {}).get('positive_gap_abs'),
    'adapter_positive_gap_abs': (manifest.get('adapter_eval_metrics') or {}).get('positive_gap_abs'),
}

index_path = ROOT / 'run_index.csv'
existing = pd.read_csv(index_path) if index_path.exists() else pd.DataFrame()
updated = pd.concat([existing, pd.DataFrame([summary])], ignore_index=True)
updated.to_csv(index_path, index=False)

print('\nSaved adapter to:', LORA_DIR)
display(pd.DataFrame([summary]))


In [ ]:
plot_paths = [
    RUN_DIR / 'plot_dpo_loss.png',
    RUN_DIR / 'plot_pair_diagnostics.png',
    RUN_DIR / 'plot_eval_base_vs_adapter_counts.png',
    RUN_DIR / 'plot_fairface_iat_word_distribution_base_vs_adapter.png',
]
for plot_path in plot_paths:
    if plot_path.exists():
        display(Markdown(f'### {plot_path.name}'))
        display(Image(filename=str(plot_path)))

print('Available adapter folders:')
for path in sorted(LORA_WEIGHTS_DIR.iterdir()):
    if path.is_dir():
        print('-', path.name)
